In [14]:
import os
from dotenv import load_dotenv
print(load_dotenv())


True


In [15]:
API_KEY = os.getenv("API_KEY")

In [16]:
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = API_KEY

llm = init_chat_model(
    "llama-3.3-70b-versatile", 
    model_provider="groq",
    temperature=0
)

In [17]:
from langchain_community.document_loaders import PyPDFLoader

loader1 = PyPDFLoader("contract1.pdf")
loader2 = PyPDFLoader("contract2.pdf")

docs1 = loader1.load()
docs2 = loader2.load()

# Add source tagging
for doc in docs1:
    doc.metadata["source"] = "Contract A"

for doc in docs2:
    doc.metadata["source"] = "Contract B"

documents = docs1 + docs2

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = splitter.split_documents(documents)

In [19]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# Example
texts = ["This is contract A clause", "This is contract B clause"]

embeddings = model.encode(texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [25]:
from sentence_transformers import SentenceTransformer

class CustomEmbeddings:
    def __init__(self):
        self.model = model
    
    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()
    
    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()
    
    def __call__(self, text):
        return self.embed_query(text)

In [26]:
from langchain_community.vectorstores import FAISS

embeddings = CustomEmbeddings()

vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever()

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [22]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [23]:
query = "Compare risks between Contract A and Contract B"

In [27]:
relevant_docs = retriever.invoke(query)

In [28]:
context = "\n\n".join([
    f"{doc.metadata.get('source', 'Unknown')}:\n{doc.page_content}"
    for doc in relevant_docs
])

In [30]:
prompt = f"""
You are a legal expert.

Compare Contract A and Contract B based on the context.

Identify:
- Risks
- Conflicts
- Liability issues
- Which contract is riskier and why

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

print(response.content)

After analyzing both contracts, I have identified the following risks, conflicts, and liability issues:

**Risks:**

1. **Confidentiality Breach**: Both contracts have confidentiality obligations, but Contract B has a broader definition of confidential information, which may increase the risk of unintentional disclosure.
2. **Limitation of Liability**: Both contracts have limitation of liability clauses, but Contract B's clause is more comprehensive, excluding liability for a wider range of damages (consequential, indirect, direct, exemplary, special, punitive, or incidental).
3. **Termination**: Both contracts have termination clauses, but Contract B's clause is more restrictive, requiring both parties to maintain confidentiality for 2 years after termination, regardless of the reason for termination.
4. **Indemnification**: Neither contract has an explicit indemnification clause, which may leave parties exposed to potential claims.

**Conflicts:**

1. **Conflicting Confidentiality Ob